# EDA of parents in anatomy dataset

In [ ]:
# ruff: noqa: T201, T203
from __future__ import annotations

import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import logging

import pandas as pd
from dotenv import load_dotenv

from config.config import DATA_DIR
from src.formatting import format_oracle_pairs_filepath
from src.onto_access import OntologyAccess
from src.onto_object import OntologyEntryAttr
from src.utils import read_oracle_pairs

logging.getLogger().setLevel(logging.WARNING)
load_dotenv()

%load_ext autoreload
%autoreload 2

In [ ]:
DATASET_NAME = "anatomy"
SET_NAME = "human-mouse"
SOURCE_ONTOLOGY = "mouse"
TARGET_ONTOLOGY = "human"

src_onto_path = DATA_DIR / DATASET_NAME / SET_NAME / f"{SOURCE_ONTOLOGY}.owl"
tgt_onto_path = DATA_DIR / DATASET_NAME / SET_NAME / f"{TARGET_ONTOLOGY}.owl"

onto_src = OntologyAccess(src_onto_path, annotate_on_init=True)
onto_tgt = OntologyAccess(tgt_onto_path, annotate_on_init=True)

In [ ]:
candidate_pairs_lines = pd.DataFrame(
    read_oracle_pairs(format_oracle_pairs_filepath(DATASET_NAME, SET_NAME)), columns=["source", "target"]
)

In [ ]:
candidate_pairs_lines.head()

In [ ]:
for col in candidate_pairs_lines.columns:
    print(f"# Total elements {col}: {candidate_pairs_lines[col].shape[0]}")
    print(f"# Duplicates {col}: {candidate_pairs_lines[col].duplicated().sum()}")
    print(f"# Unique elements {col}: {candidate_pairs_lines[col].nunique()}")

### Restructure source - target ontologies

In [ ]:
def get_df(rows: list[str], onto: OntologyAccess) -> pd.DataFrame:
    data = [
        {
            "entry": row,
            "all_entity_names": OntologyEntryAttr(row, onto).get_all_entity_names(),
            "parents_preferred_names": OntologyEntryAttr(row, onto).get_parents_preferred_names(),
            "direct_parents": OntologyEntryAttr(row, onto).get_direct_parents(),
        }
        for row in rows
    ]
    data_df = pd.DataFrame(data)
    data_df["parents_preferred_names"] = data_df["parents_preferred_names"].apply(lambda x: list(map(str, x)))
    data_df["direct_parents"] = data_df["direct_parents"].astype(str)
    return data_df


tgt_df = get_df(candidate_pairs_lines.iloc[:, 0], onto_tgt)
src_df = get_df(candidate_pairs_lines.iloc[:, 1], onto_src)

In [ ]:
src_df.head()

In [ ]:
tgt_df.head()

<b><i> Example ontology</i></b>:

<b> Number </b>: 315

<b> Column name </b>: http://mouse.owl#MA_0000293

<b> All entity names </b>: {'pelvic girdle bone'}

<b> Parents preffered names </b>: "[{'pelvis bone'}, {'trunk bone'}, {'bone'}, {'abdomen/pelvis/perineum bone'}, {'girdle bone'}, {owl.Thing}]"

<b> Direct parents </b>: "{mouse.MA_0000291, mouse.MA_0000532}"

In [ ]:
mouse_example = OntologyEntryAttr("http://mouse.owl#MA_0000293", onto_src)

parent = next(iter(mouse_example.get_direct_parents()))
child = next(iter(parent.get_direct_children()))

mouse_example, parent, child